# NYC Mesh × Weather — modular analysis notebook

End-to-end correlation of **NYC Mesh wireless link RSL** with **ASOS / PWS / Mesonet weather**.

Every plot cell exposes a small set of **knobs at the top** — edit them, re-run, no other coupling.

| knob | meaning | example values |
|------|---------|----------------|
| `PERIOD`   | which timestamps to use            | `'all'`, `'rainy'`, `'snow'`, `'dry'`, `('2026-02-01','2026-02-05')` |
| `LINKS`    | which mesh links to include        | `'all'`, `20` (top-N by coverage), `('band','5GHz')`, `('device_pattern','lbe-')`, `('top_valid', 30, 'rsl_remote')`, `['cml_id_a', ...]` |
| `BAND`     | which RF band                      | `'5GHz'`, `'60GHz'`, `'all'` |
| `END`      | which side of the link             | `'local'`, `'remote'`, `'both'` |
| `TIME_RES` | resampling resolution              | `'5min'`, `'1h'`, `'1D'` (signal native is 8 s) |
| `WEATHER_NETWORKS` | which weather networks     | `'all'`, `['ASOS']`, `['ASOS','Mesonet']` |
| `WEATHER_AREA`     | which stations within network | `'all'`, `['KJFK','KLGA','KNYC']`, `('bbox',(40.7,40.85),(-74.0,-73.9))`, `('nearest', 5, (40.81,-73.96))` |

All selection logic lives in `build_signal()` and `build_weather()` from `analysis.nycmesh_utils`.

## §1. Setup — imports + paths + global window

These are the only **global** constants. Everything else is per-cell.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160})

REPO_ROOT = Path('../../../').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from analysis.netcdf_utils import load_pws_grouped
from analysis.pws_qc import (
    read_qc_status, network_resample, filter_stations_by_bbox, DEFAULT_COLORS,
)
from analysis import nycmesh_utils as WN

# ── Master analysis window ────────────────────────────────────────────────
GLOBAL_START = pd.Timestamp('2024-07-01')
GLOBAL_END   = pd.Timestamp('2026-04-24')

# ── File paths ────────────────────────────────────────────────────────────
RAW_FULL = REPO_ROOT / 'dataset' / 'raw' / 'full'
OUT      = RAW_FULL / 'outputs'

# Preferred QC'd weather files (fall back to network files if not present).
ASOS_NC = OUT / 'asos_2023-10-01_2026-04-23.nc'
PWS_NC  = OUT / 'pws_wu_merged_2023-10-29_2026-04-24_qc.nc'
MESO_NC = OUT / 'mesonet_2023-08-01_2026-03-04.nc'
if not ASOS_NC.exists(): ASOS_NC = RAW_FULL / 'asos_nyc_network.nc'
if not PWS_NC.exists():  PWS_NC  = RAW_FULL / 'pws_wu_network.nc'

SIGNAL_NC = RAW_FULL / 'nycmesh_data_20231029_to_20260430.nc'
# Per-cml_id physical + activity metadata. cml_ids in this file match the big
# netCDF, so joins onto LINK_DF actually populate (unlike links_metadata.csv).
META_CSV  = REPO_ROOT / 'dataset' / 'meta' / 'links_metadata_mapped.csv'

# Figures dir for paper-quality saves
FIG_DIR = REPO_ROOT / 'dataset' / 'examples' / 'nycmesh_weather_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Window : {GLOBAL_START.date()} → {GLOBAL_END.date()}  ({(GLOBAL_END-GLOBAL_START).days} days)')
for f in [ASOS_NC, PWS_NC, MESO_NC, SIGNAL_NC, META_CSV]:
    mark = '✓' if f.exists() else '✗'
    sz = f.stat().st_size / 1e6 if f.exists() else 0
    print(f'  {mark} {f.name:55s} {sz:8.0f} MB')

## §2. Load weather networks

Builds the `NETWORKS` dict — `{network_name: {station_id: xr.Dataset}}` — consumed by `build_weather()`.

In [ ]:
NETWORKS = WN.load_weather_networks(asos_nc=ASOS_NC, pws_nc=PWS_NC, meso_nc=MESO_NC)

## §3. Load NYC Mesh signal data + link table

`SIGNAL` is the lazy xarray Dataset. `LINK_DF` is the per-link inventory used by `LINKS=` knobs (coverage fractions per RSL variable, derived band, optional physical metadata).

In [ ]:
SIGNAL = WN.open_nycmesh(SIGNAL_NC)
print(f'Signal: {dict(SIGNAL.sizes)}  vars={list(SIGNAL.data_vars)}')

# Per-link coverage (decimated scan; ~30–60 s, run once).
LINK_DF = WN.link_table(SIGNAL, stride=200)

# Physical + activity metadata per cml_id (length, frequency, endpoint
# lat/lon, antenna_type, coverage_days, active). cml_ids match the big netCDF
# so the merge actually populates.
META = pd.read_csv(META_CSV)
META['length_m']  = META['length']
META['freqs_MHz'] = META['frequency'].apply(lambda f: [int(f)] if pd.notna(f) else [])
LINK_DF = WN.attach_meta(LINK_DF, META.drop(columns=['device_name'], errors='ignore'))

print(f'\nLINK_DF: {LINK_DF.shape}')
print('Band distribution:')
print(LINK_DF['band'].value_counts().to_string())
print(f'\nLinks with endpoint coords: {LINK_DF["site_0_lat"].notna().sum()} / {len(LINK_DF)}')
print(f'Links currently active     : {LINK_DF["active"].sum() if "active" in LINK_DF else "n/a"}')
print('\nTop-10 links by frac_rsl_remote:')
print(LINK_DF.sort_values('frac_rsl_remote', ascending=False)
      [['cml_id','device_name','frac_rsl_remote','frac_rsl_60g_remote','band','length_m']]
      .head(10).to_string(index=False))

## §4. Basic weather stats — dry / wet / rainy / snow days

Computed once against ASOS (median across stations) and Mesonet snow depth.
`rainy_days`, `snow_days`, `dry_days` feed `PERIOD='rainy'/'snow'/'dry'` masking everywhere downstream.

In [ ]:
# ── Thresholds ────────────────────────────────────────────────────────────
RAINY_DAY_MM    = 10.0   # daily rain >= this → 'rainy' day
DRY_DAY_MM      = 0.1    # daily rain  < this → 'dry'   day
SNOW_DELTA_CM   = 2.0    # snow_depth daily Δ >= this → 'snow' day
# ──────────────────────────────────────────────────────────────────────────

classes = WN.classify_weather_days(
    NETWORKS['ASOS'], window=(GLOBAL_START, GLOBAL_END),
    rainy_threshold_mm=RAINY_DAY_MM, dry_threshold_mm=DRY_DAY_MM,
    mesonet_dict=NETWORKS['Mesonet'], snow_delta_threshold=SNOW_DELTA_CM,
)
daily_rain    = classes['daily_rain']
rainy_days    = classes['rainy_days']
dry_days      = classes['dry_days']
any_rain_days = classes['any_rain_days']
snow_days     = classes['snow_days']

summary = pd.DataFrame([{
    'window_days'                       : int(daily_rain.notna().sum()),
    f'dry_days (<{DRY_DAY_MM}mm)'       : int(len(dry_days)),
    f'any_rain (>={DRY_DAY_MM}mm)'      : int(len(any_rain_days)),
    f'wet_days (>={RAINY_DAY_MM:.0f}mm)': int(len(rainy_days)),
    f'snow_days (Δ>={SNOW_DELTA_CM:.0f})': int(len(snow_days)),
}])
print(summary.to_string(index=False))
print(f'\nTop rainy days:'); print(rainy_days.head(5).round(1).to_string())
print(f'\nTop snow days:');  print(snow_days.head(5).round(1).to_string() if len(snow_days) else '  (none)')

# Quick visual: daily rain timeline with rainy/snow days highlighted
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.bar(daily_rain.index, daily_rain.values, width=0.9, color='steelblue', alpha=0.7, label='daily rain (mm)')
ax.scatter(rainy_days.index, rainy_days.values, color='red',  s=14, zorder=5, label=f'rainy (≥{RAINY_DAY_MM:.0f} mm)')
if len(snow_days):
    ax.scatter(snow_days.index, [daily_rain.max()*0.95]*len(snow_days),
               color='cyan', marker='v', s=30, zorder=5, label=f'snow (Δ≥{SNOW_DELTA_CM:.0f})')
ax.set_ylabel('daily rain (mm)')
ax.set_title('Daily rainfall (ASOS median) — rainy & snow days highlighted')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
for lab in ax.get_xticklabels(): lab.set_rotation(30)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## §5. Link inventory — coverage by band

Bar charts of per-link coverage for each RSL variable. Use this to decide on a sensible `LINKS=` selection (e.g. `('valid_min', 0.10, 'rsl_remote')` to keep only links with ≥10 % coverage).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6), sharey=True)
for ax, col in zip(axes.flat, ['frac_rsl', 'frac_rsl_remote', 'frac_rsl_60g', 'frac_rsl_60g_remote']):
    v = LINK_DF[col].sort_values(ascending=False).reset_index(drop=True)
    ax.bar(range(len(v)), v.values, color='#1f77b4', width=1.0)
    ax.set_title(f'{col}   median={v.median():.3f}  links≥10%={(v>=0.10).sum()}')
    ax.set_xlabel('link (sorted)'); ax.set_ylabel('valid fraction'); ax.grid(True, alpha=0.3)
fig.suptitle(f'Per-link coverage  (n={len(LINK_DF)} links)', y=1.02)
plt.tight_layout(); plt.show()

# Quick text inventory of useful selection groups
groups = {
    '≥10% rsl_remote (5GHz)'      : ('valid_min', 0.10, 'rsl_remote'),
    '≥5%  rsl_60g_remote (60GHz)' : ('valid_min', 0.05, 'rsl_60g_remote'),
    'band=5GHz (auto)'            : ('band', '5GHz'),
    'band=60GHz (auto)'           : ('band', '60GHz'),
    'device contains "lbe-"'      : ('device_pattern', r'lbe-'),
    'device contains "af60"'      : ('device_pattern', r'af60'),
}
for name, spec in groups.items():
    n = len(WN.select_links(LINK_DF, spec))
    print(f'  {n:4d}  {name}')

## §5b. Link map — geographic view (with optional weather overlay)

Plots NYC Mesh links as line segments on lat/lon axes. Endpoints come from `links_metadata_mapped.csv` (loaded into `META` in §3). Use the knobs below to filter by area / band / minimum coverage, optionally restrict to the top-N best-covered links, and pick which weather networks (if any) to overlay as triangle markers.

Useful AREA recipes:
* whole NYC default:  `'all'`
* Manhattan-ish bbox: `('bbox', (40.70, 40.84), (-74.02, -73.92))`
* k closest to a hub: `('nearest', 80, (40.81, -73.96))`

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
AREA            = 'all'             # 'all' | ('bbox',(lat0,lat1),(lon0,lon1)) | ('nearest', k, (lat0,lon0))
N_LINKS         = None              # None = all matching; int = top-N by coverage_days
BAND_FILTER     = 'all'             # 'all' | '5GHz' | '24GHz' | '60GHz'
MIN_COVERAGE_D  = 0                 # drop links with fewer than this many valid days
ACTIVE_ONLY     = True              # restrict to currently active links
OVERLAY_WEATHER = ['ASOS', 'Mesonet']   # weather networks to overlay; [] = none
COLOR_BY        = 'band'            # 'band' | 'coverage' | 'length'
# ──────────────────────────────────────────────────────────────────────

WN.plot_link_map(
    META, networks=NETWORKS,
    area=AREA, n_links=N_LINKS, band=BAND_FILTER,
    min_coverage_days=MIN_COVERAGE_D, active_only=ACTIVE_ONLY,
    overlay=OVERLAY_WEATHER, color_by=COLOR_BY,
)

## §6. Signal time-series — per-link RSL

First proper knobs cell. Default selection is the **top-20 best-covered 5 GHz links** over the full period at hourly resolution. Edit the knobs to zoom or change band:

* Feb event zoom:           `PERIOD=('2026-02-01','2026-02-05')`, `TIME_RES='5min'`
* Rainy days only (mask):  `PERIOD='rainy'`
* 60 GHz top-10 links:      `BAND='60GHz'`, `LINKS=('top_valid', 10, 'rsl_60g_remote')`
* All 5 GHz links ≥10 %:    `LINKS=('valid_min', 0.10, 'rsl_remote')`

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
PERIOD   = 'all'                                # 'all' | 'rainy' | 'snow' | 'dry' | (start, end)
LINKS    = ('top_valid', 20, 'rsl_remote')      # see §5 examples
BAND     = '5GHz'                               # '5GHz' | '60GHz' | 'all'
END      = 'remote'                             # 'local' | 'remote' | 'both'
TIME_RES = '1h'
AGG      = 'mean'
# ──────────────────────────────────────────────────────────────────────

WN.plot_signal_timeseries(
    SIGNAL, LINK_DF,
    band=BAND, end=END, links=LINKS, period=PERIOD,
    time_res=TIME_RES, agg=AGG,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
)

## §7. Empirical CDF of RSL — dry vs rainy comparison

Two CDFs per panel — same set of links, but conditioned on **dry** vs **rainy** days. The shift between the two curves quantifies rain-attenuation.

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
LINKS    = ('top_valid', 50, 'rsl_remote')      # keep small for fast iteration
BAND     = 'all'                                # '5GHz' | '60GHz' | 'all'
END      = 'remote'
TIME_RES = '1h'
AGG      = 'mean'
PAD_DAYS = 0                                    # 0 = exact day match
# ──────────────────────────────────────────────────────────────────────

WN.plot_ecdf_dry_vs_rainy(
    SIGNAL, LINK_DF,
    band=BAND, end=END, links=LINKS,
    time_res=TIME_RES, agg=AGG, pad_days=PAD_DAYS,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
)

## §8. Scatter — signal vs rainfall

Per-timestamp pairing of the network-median rainfall against the link-median RSL (raw) or fade (RSL drop from rolling-95th-percentile baseline). Slope, intercept, R² and Pearson r reported in the legend — these are the headline numbers for the paper.

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
PERIOD           = 'all'                            # 'all' | 'rainy' | 'snow' | 'dry' | (start, end)
LINKS            = ('top_valid', 50, 'rsl_remote')
BAND             = 'all'                            # 'all' makes one scatter per band
END              = 'remote'
TIME_RES         = '1h'
WEATHER_NETWORKS = ['ASOS']                         # any subset of NETWORKS keys
WEATHER_AREA     = 'all'                            # 'all' | ['KJFK', ...] | ('bbox',(lat0,lat1),(lon0,lon1)) | ('nearest', k, (lat0,lon0))
WEATHER_VAR      = 'rainfall_amount'
USE_FADE         = True                             # True → RSL drop (dB), False → raw RSL (dBm)
FADE_WINDOW      = '24h'
FADE_QUANTILE    = 0.95
# ──────────────────────────────────────────────────────────────────────

WN.plot_signal_vs_rain(
    SIGNAL, LINK_DF, NETWORKS,
    band=BAND, end=END, links=LINKS, period=PERIOD, time_res=TIME_RES,
    weather_networks=WEATHER_NETWORKS, weather_area=WEATHER_AREA,
    weather_var=WEATHER_VAR,
    use_fade=USE_FADE, fade_window=FADE_WINDOW, fade_quantile=FADE_QUANTILE,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
)

## §9. Pearson correlation matrix — RSL vs weather variables

Network-median signal/fade against multiple weather features at once: rainfall, temperature, dew_point, humidity. Useful for the paper to show that rainfall dominates and other variables are essentially uncorrelated.

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
PERIOD           = 'all'
LINKS            = ('top_valid', 50, 'rsl_remote')
BAND             = 'all'
END              = 'remote'
TIME_RES         = '1h'
WEATHER_NETWORKS = ['ASOS']
WEATHER_AREA     = 'all'
WEATHER_VARS     = [('rainfall_amount', 'sum'),    # (var, agg) pairs; missing vars are skipped
                    ('temperature',     'mean'),
                    ('dewpoint',        'mean'),
                    ('wind_velocity',   'mean')]
USE_FADE         = True
# ──────────────────────────────────────────────────────────────────────

WN.plot_signal_weather_corr(
    SIGNAL, LINK_DF, NETWORKS,
    weather_vars=WEATHER_VARS,
    band=BAND, end=END, links=LINKS, period=PERIOD, time_res=TIME_RES,
    weather_networks=WEATHER_NETWORKS, weather_area=WEATHER_AREA,
    use_fade=USE_FADE,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
)

## §10. Event zoom — rain + signal stack

A single multi-panel figure for a chosen rain (or snow) event: rainfall (top), signal fade (middle), raw RSL bundle (bottom). Set `PERIOD` to a `(start, end)` window. Try the top rainy day from §4.

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
PERIOD           = ('2026-02-01', '2026-02-05')    # (start, end) | 'auto-rainy' picks the top rainy day
LINKS            = ('top_valid', 30, 'rsl_remote')
BAND             = '5GHz'
END              = 'remote'
TIME_RES         = '15min'
WEATHER_NETWORKS = 'all'                            # 'all' | ['ASOS', 'WU PWS', ...]
WEATHER_AREA     = 'all'
# ──────────────────────────────────────────────────────────────────────

WN.plot_event_zoom(
    SIGNAL, LINK_DF, NETWORKS,
    period=PERIOD, links=LINKS, band=BAND, end=END, time_res=TIME_RES,
    weather_networks=WEATHER_NETWORKS, weather_area=WEATHER_AREA,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
    network_colors=DEFAULT_COLORS,
)

## §11. Per-link statistics — paper-ready table

One row per link with raw RSL stats and rainy-vs-dry comparison. The full DataFrame is returned as `result['stats']`. Pass `save_table_path=...` / `save_hist_path=...` to `WN.per_link_stats(...)` if you want to write to disk.

In [ ]:
# ─── Knobs ────────────────────────────────────────────────────────────
LINKS    = ('top_valid', 50, 'rsl_remote')
BAND     = '5GHz'
END      = 'remote'
TIME_RES = '1h'
# ──────────────────────────────────────────────────────────────────────

result = WN.per_link_stats(
    SIGNAL, LINK_DF,
    links=LINKS, band=BAND, end=END, time_res=TIME_RES,
    window=(GLOBAL_START, GLOBAL_END),
    rainy_days=rainy_days, snow_days=snow_days, dry_days=dry_days,
)
stats = result['stats']  # full table available for further inspection

## §12. Continue from here

The notebook ends here, but the helper layer (`analysis/nycmesh_utils.py`) and the
`build_signal()` / `build_weather()` pattern are easy to extend. Some suggested next
analyses:

* **Lag analysis** — `pair_signal_weather` followed by `pd.Series.shift(...)` and
  re-correlate to find the lead/lag of fade vs rainfall.
* **Per-event integrated rain vs total fade** — group rainy days and compute
  `sum(fade)` against `sum(rain)` per event, one point per storm.
* **Band comparison** — re-run §8 with `BAND='5GHz'` and `BAND='60GHz'` side-by-side
  to quantify the higher rain sensitivity of the 60 GHz band.
* **Spatial colour map** — for links with metadata (`META`), colour-code the
  per-link rain effect on the station map.
* **CFAD (contoured frequency-by-attenuation diagram)** — 2-D histogram of fade
  (y) vs rain rate (x), with frequency contours. A staple plot for CML-rain papers.